# Online Retail - Exploratory Data Analysis

**Author:** Me :)

**Dataset:** Online Retail II (UCI Machine Learning Repository)

## About the dataset
This dataset contains transactions from a UK-based online shop between December 2009 and December 2011. The shop mainly sells gifts and home items. Some customers are wholesalers.

## Project overview
This is the first part of my E-Commerce RFM Analysis project. In this notebook, I look at the data, clean it, and explore some basic sales patterns. I will use the cleaned data for customer segmentation in Phase 2.

### What I will do
1. Load and inspect the data
2. Clean the data
3. Explore sales, products, countries, and customers
4. Save the cleaned data for the next phase

---

## Setup

First, import the libraries I will use in this notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Set a consistent style for the charts
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

TEAL = "#1D9E75"
CORAL = "#D85A30"
PURPLE = "#534AB7"
AMBER = "#BA7517"
GRAY = "#888780"


---
## 1. Load and Inspect

First, I will check the size of the dataset, the column types, and whether any values are missing.

In [ ]:
# Keep online_retail.csv in the same folder as this notebook
df = pd.read_csv("online_retail.csv")

print(f"Dataset size: {df.shape[0]:,} rows and {df.shape[1]} columns")


In [ ]:
# Check the column types
print('── Column types ──')
print(df.dtypes)

print('\n── Sample rows ──')
df.head()

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

pd.DataFrame({'Missing': missing, '% of total': missing_pct}).query('Missing > 0')

In [ ]:
# Get some basic information about the data
print(f'Date range   : {df["InvoiceDate"].min()}  →  {df["InvoiceDate"].max()}')
print(f'Customers    : {df["Customer ID"].nunique():,}')
print(f'Products     : {df["StockCode"].nunique():,}')
print(f'Countries    : {df["Country"].nunique()}')

**What I noticed:**

- Some rows are missing `Customer ID`. I will exclude these rows because I need customer IDs for customer-level analysis.
- Some rows have no product description, so I will check whether that affects the product analysis.
- `InvoiceDate` needs to be converted to a date/time type before grouping by month or day.
- The dataset covers transactions across roughly two years.


---
## 2. Data Cleaning

I will clean the data before making charts.

| Step | What I will do | Why |
|---|---|---|
| 2-A | Remove rows without a customer ID | Needed for customer-level analysis |
| 2-B | Remove duplicate rows | Avoid counting duplicate transaction lines |
| 2-C | Set cancelled invoices aside | Keep them separate from sales |
| 2-D | Remove rows with non-positive quantity or price from the sales data | Avoid including these rows in positive sales totals |
| 2-E | Fix column types | Make dates and customer IDs easier to work with |
| 2-F | Add columns for revenue and date grouping | Make the analysis easier |


In [ ]:
original_len = len(df)

# 2-A: Remove rows without a customer ID
df = df.dropna(subset=['Customer ID'])
dropped_no_id = original_len - len(df)

# 2-B: Remove duplicate rows
dups = df.duplicated().sum()
df = df.drop_duplicates()

# 2-C: Separate cancelled invoices (invoice numbers start with 'C')
# Keep these rows for the returns analysis later
cancelled_mask = df['Invoice'].astype(str).str.startswith('C')
df_cancelled   = df[cancelled_mask].copy()
df             = df[~cancelled_mask]

# 2-D: Remove rows with quantity or price less than or equal to zero
invalid_qty   = (df['Quantity'] <= 0).sum()
invalid_price = (df['Price'] <= 0).sum()
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

print('── Cleaning log ──')
print(f'  Dropped (no Customer ID)  : {dropped_no_id:,}')
print(f'  Dropped (duplicates)      : {dups:,}')
print(f'  Isolated (cancellations)  : {cancelled_mask.sum():,}')
print(f'  Dropped (invalid qty)     : {invalid_qty:,}')
print(f'  Dropped (invalid price)   : {invalid_price:,}')
print(f'\n  Rows remaining : {len(df):,}')

In [ ]:
# 2-E: Change column types

# Change Customer ID to a string
df['Customer ID'] = df['Customer ID'].astype(int).astype(str)

# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print('Types after fix:')
print(df[['Customer ID', 'InvoiceDate']].dtypes)

In [ ]:
# 2-F: Add columns for the analysis

df['TotalPrice'] = df['Quantity'] * df['Price']     # revenue for each row
df['YearMonth']  = df['InvoiceDate'].dt.to_period('M')  # used to group by month
df['Month']      = df['InvoiceDate'].dt.month
df['Year']       = df['InvoiceDate'].dt.year
df['DayOfWeek']  = df['InvoiceDate'].dt.day_name()  # get the day name

print(f'Clean dataset: {len(df):,} rows  |  {df["Customer ID"].nunique():,} unique customers')
df.head(3)

---
## 3. Exploratory Data Analysis

Now I will make a few charts to understand the sales data. I will look at monthly revenue, countries, products, order days, customer revenue, and cancellations.

### 3-A - Monthly Revenue

This chart shows how revenue changes from month to month.

In [ ]:
monthly_rev = df.groupby("YearMonth")["TotalPrice"].sum().reset_index()
monthly_rev["YearMonth"] = monthly_rev["YearMonth"].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(monthly_rev["YearMonth"], monthly_rev["TotalPrice"],
        color=TEAL, linewidth=2, marker="o", markersize=4)

ax.set_title("Monthly Revenue")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue (£)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x/1000:.0f}K"))
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("01_monthly_revenue.png")
plt.show()


**What I noticed:** Revenue changes from month to month, with some higher months near the end of the year. This may be related to seasonal gift shopping, but the chart alone cannot confirm the reason.

### 3-B - Top 10 Countries (excluding the UK)

The UK has the most sales, so I am leaving it out here to compare the other countries.

In [ ]:
top_countries = (
    df[df['Country'] != 'United Kingdom']
    .groupby('Country')['TotalPrice']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(
    top_countries.index[::-1],
    top_countries.values[::-1],
    color=PURPLE, edgecolor='white', linewidth=0.5
)

# Add the revenue value to each bar
for bar in bars:
    ax.text(
        bar.get_width() + 500,
        bar.get_y() + bar.get_height() / 2,
        f'£{bar.get_width()/1000:.1f}K',
        va='center', fontsize=10
    )

ax.set_title('Top 10 Countries by Revenue (excl. UK)')
ax.set_xlabel('Revenue (£)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('02_top_countries.png')
plt.show()

**What I noticed:** EIRE has the highest revenue among the countries shown outside the UK. The Netherlands and Germany also appear among the leading markets. This chart compares revenue, not profit or number of customers.

### 3-C - Top 10 Products by Revenue

This chart shows which products brought in the most revenue.

In [ ]:
top_products = (
    df.groupby('Description')['TotalPrice']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(
    top_products.index[::-1],
    top_products.values[::-1],
    color=AMBER, edgecolor='white', linewidth=0.5
)

for bar in bars:
    ax.text(
        bar.get_width() + 200,
        bar.get_y() + bar.get_height() / 2,
        f'£{bar.get_width()/1000:.1f}K',
        va='center', fontsize=10
    )

ax.set_title('Top 10 Products by Revenue')
ax.set_xlabel('Revenue (£)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('03_top_products.png')
plt.show()

**What I noticed:** REGENCY CAKESTAND 3 TIER is among the highest-revenue products in this dataset. It could be useful to check how often it is purchased and by how many customers.

### 3-D - Orders by Day of the Week

This chart compares the number of orders on each day.

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

orders_by_day = (
    df.groupby('DayOfWeek')['Invoice']
    .nunique()
    .reindex(day_order)
)

fig, ax = plt.subplots(figsize=(8, 4))

# Use a different colour for Sunday
colors = [TEAL if d != 'Sunday' else GRAY for d in day_order]
ax.bar(orders_by_day.index, orders_by_day.values, color=colors, edgecolor='white')

ax.set_title('Number of Orders by Day of Week')
ax.set_xlabel('Day')
ax.set_ylabel('Unique Orders')

plt.tight_layout()
plt.savefig('04_orders_by_day.png')
plt.show()

**What I noticed:** The order counts differ by day of the week. Thursday appears busy in this dataset, while Sunday has very few or no recorded orders. This describes the data and does not explain why the pattern occurs.

### 3-E - Revenue per Customer

Customer revenue varies a lot, so I use `log(Revenue + 1)` to make the differences easier to see in the histogram.

In [ ]:
customer_rev = df.groupby('Customer ID')['TotalPrice'].sum()

fig, ax = plt.subplots(figsize=(8, 4))

# Use log(Revenue + 1) for the histogram
ax.hist(np.log1p(customer_rev), bins=50, color=CORAL, edgecolor='white', linewidth=0.4)

ax.set_title('Revenue Distribution per Customer (log scale)')
ax.set_xlabel('log(Revenue + 1)')
ax.set_ylabel('Number of Customers')

plt.tight_layout()
plt.savefig('05_revenue_distribution.png')
plt.show()

**What I noticed:** Most customers bring in less revenue, while a smaller group brings in much more. This is one reason customer segmentation could be useful in Phase 2.

### 3-F - Returns Analysis

I separated invoices starting with `C` earlier. Here I am checking how many invoices were cancelled.

In [ ]:
# Count unique invoices in the cleaned sales data and in the cancelled invoices
total_invoices = df["Invoice"].nunique()
total_cancelled = df_cancelled["Invoice"].nunique()

# This is a simple invoice cancellation share, not a confirmed product return rate
all_invoices = total_invoices + total_cancelled
cancellation_rate = total_cancelled / all_invoices * 100 if all_invoices else 0

print(f"Sales invoices      : {total_invoices:,}")
print(f"Cancelled invoices  : {total_cancelled:,}")
print(f"Cancellation share  : {cancellation_rate:.1f}%")


**What I noticed:** Some invoices were cancelled. This calculation compares the number of cancelled invoices with the number of sales and cancelled invoices counted here. It should be treated as a simple cancellation share, not a complete returns rate, because product returns and cancellations are not necessarily the same thing.

---
## 4. Export

Save the cleaned data to a CSV file so I can use it in Phase 2.

In [ ]:
OUTPUT_PATH = 'retail_clean.csv'

df.to_csv(OUTPUT_PATH, index=False)

print(f'Saved → {OUTPUT_PATH}')
print(f'Shape : {df.shape[0]:,} rows × {df.shape[1]} columns')

---
## Summary - Key Metrics

These metrics give a quick overview of the cleaned sales data.

In [ ]:
total_revenue = df["TotalPrice"].sum()
total_orders = df["Invoice"].nunique()
total_customers = df["Customer ID"].nunique()
average_order_value = total_revenue / total_orders if total_orders else 0

best_month = (
    df.groupby("YearMonth")["TotalPrice"]
    .sum()
    .idxmax()
)

top_country = (
    df[df["Country"] != "United Kingdom"]
    .groupby("Country")["TotalPrice"]
    .sum()
    .idxmax()
)

top_product = (
    df.groupby("Description")["TotalPrice"]
    .sum()
    .idxmax()
)

print(f"Total revenue: £{total_revenue:,.2f}")
print(f"Number of orders: {total_orders:,}")
print(f"Number of customers: {total_customers:,}")
print(f"Average order value: £{average_order_value:,.2f}")
print(f"Highest-revenue month: {best_month}")
print(f"Top country outside the UK: {top_country}")
print(f"Top product by revenue: {top_product}")


---

## Limitations and Next Steps

A few things to keep in mind:

- Rows without a customer ID are excluded, so customer-level results do not cover every transaction in the raw data.
- Rows with non-positive quantity or price are excluded from the sales analysis. These may include adjustments that could be worth studying separately.
- Cancelled invoices are counted separately; the cancellation share is not the same as a complete product returns rate.
- The charts show patterns in the dataset, but they do not prove why those patterns happen.

**Phase 2 - RFM Scoring**

In the next part, I will use `retail_clean.csv` to:
1. Calculate Recency, Frequency, and Monetary values for each customer.
2. Give each value a score.
3. Group customers into useful segments.
4. Prepare the results for a Power BI dashboard.
